# Anomaly Detection Model Training

This notebook trains an unsupervised anomaly detection model on the synthetic Rx rebate dataset  
and evaluates how well it surfaces known injected anomalies.

**Pipeline overview:**
1. Load invoice data (with injected anomalies)
2. Feature engineering at the NDC × client × quarter grain
3. Build ground-truth labels from known injection patterns
4. Train `IsolationForest` with `RobustScaler`
5. Combine ML scores with rule-based flags into a priority score
6. Evaluate: Precision@K and Dollars Captured@K
7. Build an audit queue (top-50) with anomaly highlights

**Domain context:** In rebate recovery, the goal is to rank invoice rows so that  
auditors review the most impactful anomalies first.  
Precision@K answers: "Of the top K rows I review, what fraction are real anomalies?"

## 1. Setup & Data Load

In [ ]:
import polars as pl
import numpy as np
import altair as alt
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Reproducibility
SEED = 42
np.random.seed(SEED)

alt.renderers.enable("default")

data_dir = Path("../data/synthetic")

invoices  = pl.read_parquet(data_dir / "invoices_with_anomalies.parquet")
drugs     = pl.read_parquet(data_dir / "drugs.parquet")
contracts = pl.read_parquet(data_dir / "contracts.parquet")

print(f"Invoices loaded : {len(invoices):,} rows")
print(f"Drugs loaded    : {len(drugs):,} rows")
print(f"Contracts loaded: {len(contracts):,} rows")
print()
print("Invoice columns:", invoices.columns)

## 2. Feature Engineering

The invoice table is already at NDC × client × quarter grain.  
We compute derived features that capture the core signals for rebate leakage:

| Feature | Signal |
|---------|--------|
| `rebate_realization` | How close actual_rebate is to expected; < 0.95 is suspicious |
| `rebate_gap` | Absolute dollar shortfall (expected − actual) |
| `rebate_per_util` | Expected rebate per invoiced unit; outliers suggest unit errors |
| `dispute_ratio` | Fraction of actual rebate that is disputed |
| `paid_ratio` | Fraction of actual rebate that was actually paid |
| `is_unmapped_ndc` | Binary: NDC absent from the drug master |

In [ ]:
# Work with non-zero expected rebate rows only (zero-rebate rows carry no signal)
inv = invoices.filter(pl.col("expected_rebate") > 0).clone()

drug_ndcs = set(drugs["ndc11"].to_list())

inv = inv.with_columns([
    # Core realization metrics
    (pl.col("actual_rebate") / (pl.col("expected_rebate") + 1e-9))
        .alias("rebate_realization"),
    (pl.col("expected_rebate") - pl.col("actual_rebate"))
        .alias("rebate_gap"),
    # Unit-level rate (detects unit-conversion errors)
    (pl.col("expected_rebate") / (pl.col("invoiced_utilization") + 1e-9))
        .alias("rebate_per_util"),
    # Dispute and payment ratios
    (pl.col("disputed_rebate") / (pl.col("actual_rebate") + 1e-9))
        .alias("dispute_ratio"),
    (pl.col("paid_rebate") / (pl.col("actual_rebate") + 1e-9))
        .alias("paid_ratio"),
    # Unmapped NDC flag (1 = NDC not in drug master)
    (~pl.col("ndc11").is_in(list(drug_ndcs))).cast(pl.Int32)
        .alias("is_unmapped_ndc"),
])

print(f"Feature-engineered rows: {len(inv):,}")
print()
print("Sample feature values:")
print(inv.select([
    "ndc11", "client_id", "invoice_quarter",
    "rebate_realization", "rebate_gap", "rebate_per_util",
    "dispute_ratio", "paid_ratio", "is_unmapped_ndc",
]).head(5))

## 3. QoQ Change Features (Time-Series Signal)

Quarter-over-quarter changes in realization flag sudden drops in payment behavior.  
A manufacturer paying 98% → 0% in one quarter is a strong anomaly signal.  

We compute the per-NDC × client rolling mean realization and the deviation from that mean.

In [ ]:
# Sort by NDC, client, then quarter so lag operations are well-ordered
inv = inv.sort(["ndc11", "client_id", "invoice_quarter"])

# For each (ndc11, client_id) group, compute the rolling mean realization
# and the deviation of the current row from that rolling mean.
# Polars uses shift() for lag, and rolling_mean() over partitioned groups.
inv = inv.with_columns(
    pl.col("rebate_realization")
      .shift(1)
      .over(["ndc11", "client_id"])
      .alias("prev_quarter_realization")
).with_columns(
    (pl.col("rebate_realization") - pl.col("prev_quarter_realization"))
      .alias("qoq_realization_change")
)

# Rolling 2-quarter mean (includes current); min_samples=1 avoids null on first row
inv = inv.with_columns(
    pl.col("rebate_realization")
      .rolling_mean(window_size=2, min_samples=1)
      .over(["ndc11", "client_id"])
      .alias("rolling2q_mean_realization")
)

print("QoQ feature sample (sorted by qoq_realization_change ascending):")
print(
    inv.filter(pl.col("qoq_realization_change").is_not_null())
       .sort("qoq_realization_change")
       .select([
           "invoice_quarter", "ndc11", "client_id",
           "rebate_realization", "prev_quarter_realization", "qoq_realization_change",
       ])
       .head(8)
)

## 4. Build Ground-Truth Labels

We reconstruct labels directly from the data using the same rules as the anomaly injectors.  
These labels are used **only for evaluation** — the model trains without seeing them.

In [ ]:
# MISSING_REBATE: expected > 0, actual = 0, NDC in drug master
missing_labels = (
    inv.filter(
        (pl.col("actual_rebate") == 0)
        & (pl.col("is_unmapped_ndc") == 0)  # exclude unmapped NDC (different anomaly type)
    )
    .select(["invoice_quarter", "ndc11", "client_id", "expected_rebate"])
    .with_columns([
        pl.lit("MISSING_REBATE").alias("anomaly_type"),
        pl.col("expected_rebate").alias("estimated_impact"),
    ])
)

# UNMAPPED_NDC: NDC not in drug master
unmapped_labels = (
    inv.filter(pl.col("is_unmapped_ndc") == 1)
    .select(["invoice_quarter", "ndc11", "client_id", "expected_rebate"])
    .with_columns([
        pl.lit("UNMAPPED_NDC").alias("anomaly_type"),
        pl.col("expected_rebate").alias("estimated_impact"),
    ])
)

# REBATE_YIELD_COLLAPSE: realization < 50% with non-trivial expected rebate
collapse_labels = (
    inv.filter(
        (pl.col("rebate_realization") < 0.50)
        & (pl.col("actual_rebate") > 0)
        & (pl.col("expected_rebate") > 10)
    )
    .select(["invoice_quarter", "ndc11", "client_id", "expected_rebate", "actual_rebate"])
    .with_columns([
        pl.lit("REBATE_YIELD_COLLAPSE").alias("anomaly_type"),
        (pl.col("expected_rebate") - pl.col("actual_rebate")).alias("estimated_impact"),
    ])
)

# DISPUTE_SPIKE: disputed_rebate > 0
dispute_labels = (
    inv.filter(pl.col("dispute_ratio") > 0)
    .select(["invoice_quarter", "ndc11", "client_id", "disputed_rebate"])
    .with_columns([
        pl.lit("DISPUTE_SPIKE").alias("anomaly_type"),
        pl.col("disputed_rebate").alias("estimated_impact"),
    ])
)

# Combine into a single label set with consistent columns
label_cols = ["invoice_quarter", "ndc11", "client_id", "anomaly_type", "estimated_impact"]

all_labels = pl.concat([
    missing_labels.select(label_cols),
    unmapped_labels.select(label_cols),
    collapse_labels.select(label_cols),
    dispute_labels.select(label_cols),
], how="vertical")

print(f"Total labeled anomaly rows: {len(all_labels)}")
print()
print(all_labels.group_by("anomaly_type").agg([
    pl.len().alias("count"),
    pl.col("estimated_impact").sum().round(2).alias("total_impact"),
]).sort("total_impact", descending=True))

## 5. Merge Labels and Prepare Feature Matrix

We join the labels onto the invoice rows so evaluation is possible.  
Rows without a label are treated as normal (no anomaly).

In [ ]:
# Left-join labels so all invoice rows are retained
# One invoice row may have multiple anomaly types (e.g. unmapped + missing)
# — we keep the first match per row for binary classification purposes
labeled = inv.join(
    all_labels.unique(subset=["invoice_quarter", "ndc11", "client_id"]),
    on=["invoice_quarter", "ndc11", "client_id"],
    how="left",
).with_columns(
    pl.col("anomaly_type").is_not_null().alias("is_anomaly")
)

print(f"Total invoice rows (non-zero expected): {len(labeled):,}")
print(f"  Labeled anomalies : {labeled['is_anomaly'].sum():,}")
print(f"  Normal rows       : {(~labeled['is_anomaly']).sum():,}")

# Feature columns for the model
# Note: qoq_realization_change has nulls for first-quarter rows; imputer handles them
FEATURE_COLS = [
    "rebate_realization",
    "rebate_gap",
    "rebate_per_util",
    "dispute_ratio",
    "paid_ratio",
    "is_unmapped_ndc",
    "qoq_realization_change",
    "rolling2q_mean_realization",
]

X = labeled.select(FEATURE_COLS).to_numpy()
print(f"\nFeature matrix shape: {X.shape}")
print("Features:", FEATURE_COLS)

## 6. Train Isolation Forest

**IsolationForest** works well for rebate anomaly detection because:
- It is fully unsupervised — no labels needed during training
- It handles mixed-scale features gracefully with `RobustScaler`
- Its decision function returns a score: more negative = more anomalous
- `contamination` controls the threshold; we set 2% because anomalies are rare

Pipeline: `SimpleImputer` (median fill for null QoQ values) → `RobustScaler` → `IsolationForest`

In [ ]:
pipe = Pipeline([
    # Fill nulls (first-quarter rows have no prev-quarter data) with median
    ("imputer", SimpleImputer(strategy="median")),
    # RobustScaler is less sensitive to outliers than StandardScaler
    ("scaler", RobustScaler()),
    ("model", IsolationForest(
        n_estimators=300,
        contamination=0.02,   # expect ~2% anomalies in the invoice population
        max_features=1.0,
        random_state=SEED,
        n_jobs=-1,
    )),
])

pipe.fit(X)

# decision_function: lower (more negative) = more anomalous
scores = pipe.decision_function(X)
preds  = pipe.predict(X)  # -1 = predicted anomaly, +1 = normal

labeled = labeled.with_columns([
    pl.lit(scores).alias("anomaly_score"),
    pl.lit(preds == -1).alias("is_anomaly_pred"),
])

print("Model trained.")
print(f"  Predicted anomalies : {(preds == -1).sum():,} / {len(preds):,}")
print(f"  Score range         : [{scores.min():.4f}, {scores.max():.4f}]")

## 7. Rule-Based Flags

Rule-based flags are human-interpretable signals derived from domain knowledge.  
Used alongside the ML score to build a priority composite:

In [ ]:
# Compute global statistics for thresholds
mean_gap   = labeled["rebate_gap"].mean()
mean_score = labeled["anomaly_score"].mean()
std_score  = labeled["anomaly_score"].std()

labeled = labeled.with_columns([
    # realization < 90%: manufacturer paid less than 90 cents on the dollar
    (pl.col("rebate_realization") < 0.90).alias("low_realization_flag"),
    # gap above the average: this row has an above-average dollar shortfall
    (pl.col("rebate_gap") > mean_gap).alias("high_gap_flag"),
    # any portion of the rebate is in dispute
    (pl.col("disputed_rebate") > 0).alias("disputed_flag"),
    # NDC not in the drug master
    (pl.col("is_unmapped_ndc") == 1).alias("unmapped_ndc_flag"),
    # sharp QoQ drop: realization fell by more than 40 percentage points
    (pl.col("qoq_realization_change").fill_null(0) < -0.40).alias("sharp_drop_flag"),
])

# Summarise flag hit rates
flag_cols = [
    "low_realization_flag", "high_gap_flag", "disputed_flag",
    "unmapped_ndc_flag", "sharp_drop_flag",
]
for col in flag_cols:
    n = labeled[col].sum()
    pct = n / len(labeled) * 100
    print(f"  {col:<30}: {n:>6,} rows  ({pct:.2f}%)")

## 8. Priority Scoring: Combine ML + Rules

The priority score merges the ML anomaly score with rule-based flags.  
We negate `anomaly_score` first so that higher priority = more anomalous.  

**Weight rationale:**
- ML score (50%): captures complex multi-feature interactions
- Low realization (20%): direct single-metric signal
- High gap (15%): prioritizes high-dollar shortfalls
- Dispute flag (10%): confirmed contested amounts
- Unmapped NDC (5%): structural anomaly; already captured in ML

In [ ]:
# Normalize ML score to [0, 1] so it combines cleanly with binary flags
score_min = labeled["anomaly_score"].min()
score_max = labeled["anomaly_score"].max()

labeled = labeled.with_columns(
    # Invert and normalize: lower (more negative) score → higher normalized value
    ((pl.col("anomaly_score") - score_max) / (score_min - score_max + 1e-9))
        .alias("ml_score_norm")
)

labeled = labeled.with_columns(
    (
        pl.col("ml_score_norm")               * 0.50
        + pl.col("low_realization_flag").cast(pl.Float64) * 0.20
        + pl.col("high_gap_flag").cast(pl.Float64)        * 0.15
        + pl.col("disputed_flag").cast(pl.Float64)        * 0.10
        + pl.col("unmapped_ndc_flag").cast(pl.Float64)    * 0.05
    ).alias("priority_score")
)

print("Priority score distribution:")
print(labeled["priority_score"].describe())
print()
print("Priority score for labeled anomalies vs. normal rows:")
print(
    labeled.group_by("is_anomaly")
    .agg([
        pl.col("priority_score").mean().round(4).alias("mean_score"),
        pl.col("priority_score").median().round(4).alias("median_score"),
    ])
)

## 9. Evaluation: Precision@K

**Precision@K** = (true anomalies in top-K) / K

In a real deployment the top-K rows form the "audit queue" — the list sent to analysts.  
High Precision@K means analysts spend less time on false positives.

In [ ]:
def precision_at_k(scored_df: pl.DataFrame, k: int) -> float:
    """Fraction of true anomalies in the top-K rows by priority_score."""
    top_k = scored_df.sort("priority_score", descending=True).head(k)
    return top_k["is_anomaly"].mean()  # bool mean = fraction True


def recall_at_k(scored_df: pl.DataFrame, k: int) -> float:
    """Fraction of ALL true anomalies that appear in the top-K rows."""
    top_k = scored_df.sort("priority_score", descending=True).head(k)
    n_detected = top_k["is_anomaly"].sum()
    n_total = scored_df["is_anomaly"].sum()
    return n_detected / n_total if n_total > 0 else 0.0


print(f"Total anomaly rows in dataset: {labeled['is_anomaly'].sum()}")
print()
print(f"{'K':>6}  {'Precision@K':>14}  {'Recall@K':>10}")
print("-" * 36)
for k in [10, 25, 50, 100, 200]:
    p = precision_at_k(labeled, k)
    r = recall_at_k(labeled, k)
    print(f"{k:>6}  {p:>13.1%}  {r:>9.1%}")

## 10. Evaluation: Dollars Captured@K

**Dollars Captured@K** = sum of `estimated_impact` for true anomalies in the top-K rows.

This is the recovery-value metric: how much money can auditors potentially recover  
if they investigate the top-K flagged rows?

In [ ]:
def dollars_captured_at_k(scored_df: pl.DataFrame, k: int) -> float:
    """Total estimated_impact of true anomalies in the top-K rows."""
    top_k = scored_df.sort("priority_score", descending=True).head(k)
    return top_k.filter(pl.col("is_anomaly"))["estimated_impact"].sum()


total_impact = labeled.filter(pl.col("is_anomaly"))["estimated_impact"].sum()
print(f"Total recoverable impact (all labeled anomalies): ${total_impact:>10,.2f}")
print()
print(f"{'K':>6}  {'$Captured':>12}  {'% of Total':>12}")
print("-" * 36)
for k in [10, 25, 50, 100, 200]:
    dollars = dollars_captured_at_k(labeled, k)
    pct = dollars / total_impact * 100 if total_impact > 0 else 0.0
    print(f"{k:>6}  ${dollars:>11,.2f}  {pct:>11.1f}%")

## 11. Visualization: Anomaly Score Distribution

IsolationForest's `decision_function` returns lower values for outliers.  
We expect anomalies to cluster at the left (more negative) end of the distribution.

In [ ]:
# Prepare data for visualization
plot_df = labeled.select([
    "anomaly_score",
    pl.col("is_anomaly").cast(pl.Utf8).replace({"true": "Anomaly", "false": "Normal"}),
]).to_pandas()

chart = (
    alt.Chart(plot_df)
    .mark_bar(opacity=0.6)
    .encode(
        x=alt.X(
            "anomaly_score:Q",
            bin=alt.Bin(maxbins=60),
            title="Isolation Forest Anomaly Score (lower = more anomalous)",
        ),
        y=alt.Y("count():Q", title="Count"),
        color=alt.Color(
            "is_anomaly:N",
            scale=alt.Scale(domain=["Normal", "Anomaly"], range=["steelblue", "crimson"]),
            title="Row Type",
        ),
        tooltip=["is_anomaly", "count()"],
    )
    .properties(
        title="Isolation Forest Score Distribution: Normal vs. Labeled Anomalies",
        width=620, height=320,
    )
)
chart

## 12. Audit Queue: Top 50 by Priority Score

The audit queue is the list of invoice rows sent to analysts for investigation.  
Rows with `anomaly_type` are known injected anomalies (highlighted).  
Rows without `anomaly_type` are model-generated flags (may be real false positives or unlabeled anomalies).

In [ ]:
audit_queue = (
    labeled
    .sort("priority_score", descending=True)
    .head(50)
    .select([
        "invoice_quarter", "ndc11", "client_id", "manufacturer",
        "expected_rebate", "actual_rebate", "rebate_gap",
        "rebate_realization", "dispute_ratio",
        "priority_score", "anomaly_type", "is_anomaly",
    ])
    .with_columns(
        pl.col("rebate_realization").round(3),
        pl.col("priority_score").round(4),
        pl.col("rebate_gap").round(2),
    )
)

n_true_in_queue = audit_queue["is_anomaly"].sum()
print(f"Top-50 audit queue: {n_true_in_queue} known anomalies, {50 - n_true_in_queue} unlabeled/FP")
print()
print(audit_queue.to_pandas().to_string(max_rows=50, max_cols=12, index=False))

## 13. Confusion Matrix / Detection Summary

Compare how many of the labeled anomalies the model detects at various thresholds.

In [ ]:
total_anomalies = labeled["is_anomaly"].sum()

print("=== Detection Summary ===")
print(f"Total labeled anomaly rows   : {total_anomalies}")
print()

for k in [10, 25, 50, 100]:
    top_k = labeled.sort("priority_score", descending=True).head(k)
    detected     = top_k["is_anomaly"].sum()
    false_pos    = k - detected
    missed       = total_anomalies - detected
    precision_k  = detected / k
    recall_k     = detected / total_anomalies if total_anomalies > 0 else 0.0
    print(f"  Top-{k:<3}: {detected} detected, {false_pos} FP, {missed} missed  "
          f"| P={precision_k:.0%}  R={recall_k:.0%}")

print()
print("Detected anomaly types in Top-50:")
top50 = labeled.sort("priority_score", descending=True).head(50)
print(
    top50.filter(pl.col("is_anomaly"))
    .group_by("anomaly_type")
    .agg([
        pl.len().alias("detected"),
        pl.col("estimated_impact").sum().round(2).alias("impact_captured"),
    ])
    .sort("detected", descending=True)
)

## 14. Conclusions & Next Steps

In [ ]:
p_at_50  = precision_at_k(labeled, 50)
p_at_100 = precision_at_k(labeled, 100)
r_at_100 = recall_at_k(labeled, 100)
dollars_at_50 = dollars_captured_at_k(labeled, 50)

top_100 = labeled.sort("priority_score", descending=True).head(100)
detected_in_100 = top_100["is_anomaly"].sum()

print(f"""
=== Results Summary ===

Model: IsolationForest (n_estimators=300, contamination=0.02)
Features: {len(FEATURE_COLS)} (realization, gap, utilization rate, dispute/paid ratio, QoQ change)
Scoring: 50% ML score + 20% realization flag + 15% gap flag + 10% dispute flag + 5% unmapped NDC

Evaluation on synthetic labeled anomalies
  Precision@50  : {p_at_50:.1%}
  Precision@100 : {p_at_100:.1%}
  Recall@100    : {r_at_100:.1%}
  Dollars@50    : ${dollars_at_50:,.2f} (estimated recoverable in top-50 queue)
  Detected {detected_in_100} of {total_anomalies} labeled anomalies in top-100

Next Steps
  1. Validate on real data with actual rebate recovery outcome labels
  2. Tune contamination parameter using historical dispute/recovery rates
  3. Add product-level features: market share, contract vintage, tier stability
  4. Experiment with PYOD models (ECOD, COPOD) for comparison
  5. Deploy as a monthly batch pipeline scoring each new invoice quarter
  6. Build feedback loop: analyst decisions → label refinement → model retraining
""")